# Chapter 3 Hands-On: Building a Server Access Log Analyzer

You're building a small tool that ingests raw HTTP access-log events and answers operational questions about them: who accessed what, error rates per endpoint, which users share roles, and how per-request config overrides layer on top of defaults.

This is deliberately close to a real observability/access-control task. Each part exercises a different Chapter 3 idea. **Do not import pandas or any third-party library** — the whole point is to use plain dicts, sets, and the `collections` module.

Work top to bottom. Each part has a task, a place to write your solution, and (further down) assertion cells that must pass. If an assertion fails, that's a gap worth investigating — don't just patch until green, understand *why*.

**Constraints throughout:**
- Prefer comprehensions and set algebra over manual loops where natural.
- Reach for `defaultdict` / `setdefault` / `Counter` / `ChainMap` where they fit.
- No mutable global state beyond what's asked.

---

## Sample data (run this first — do not edit)

Each log line is a raw string: `timestamp user method path status`. Status is an HTTP code.

In [8]:
RAW_LOG = """\
2026-08-23T09:00:01 alice GET /api/orders 200
2026-08-23T09:00:03 bob POST /api/orders 201
2026-08-23T09:00:05 alice GET /api/orders 200
2026-08-23T09:00:07 carol GET /api/reports 500
2026-08-23T09:00:09 alice DELETE /api/orders 403
2026-08-23T09:00:11 bob GET /api/reports 200
2026-08-23T09:00:13 carol GET /api/reports 500
2026-08-23T09:00:15 dave GET /api/health 200
2026-08-23T09:00:17 alice GET /api/orders 200
2026-08-23T09:00:19 bob POST /api/orders 500
2026-08-23T09:00:21 carol GET /api/orders 404
2026-08-23T09:00:23 dave GET /api/health 200
2026-08-23T09:00:25 alice GET /api/reports 200
2026-08-23T09:00:27 eve POST /api/orders 401
"""

LOG_LINES = RAW_LOG.strip().splitlines()

# Role assignments (who belongs to which team)
ROLES = {
    "admins": {"alice", "bob"},
    "analysts": {"bob", "carol"},
    "ops": {"dave"},
    # note: eve is intentionally in no role
}

print(f"{len(LOG_LINES)} log lines loaded")
print(LOG_LINES[0])



14 log lines loaded
2026-08-23T09:00:01 alice GET /api/orders 200
2026-08-23T09:00:01 alice GET /api/orders 200


## Part 1 — Parse into records (dict comprehension + named record)

Write `parse_log(lines)` that returns a **list of records**. Each record should be an immutable, field-accessible object built from `collections.namedtuple` (or `typing.NamedTuple`) with fields: `timestamp, user, method, path, status`. `status` must be an `int`.

Then build `records = parse_log(LOG_LINES)`.

Concepts: record modeling (Ch2 connection), converting a flat line into a structured record.

In [15]:
from collections import namedtuple

Event = namedtuple("Event", "timestamp user method path status")

# for line in LOG_LINES:
#     x1, x2, x3, x4, x5 = line.split()
#     e = Event(x1, x2, x3, x4, int(x5))
#     print(e)

def parse_log(lines):
    # TODO: split each line into 5 fields, coerce status to int,
    # and return a list of Event records.
    return [Event(timestamp, user, method, path, int(status)) for timestamp, user, method, path, status in (line.split() for line in lines)]


records = parse_log(LOG_LINES)
records[:2]

[Event(timestamp='2026-08-23T09:00:01', user='alice', method='GET', path='/api/orders', status=200),
 Event(timestamp='2026-08-23T09:00:03', user='bob', method='POST', path='/api/orders', status=201)]

## Part 2 — Group requests by endpoint (defaultdict vs setdefault)

Write `requests_per_path(records)` that returns a mapping `path -> list of users who hit it (in order, with duplicates)`.

Do it **twice** in the same function body to feel the difference, then return the `defaultdict` version:
1. First accumulate using `dict.setdefault`.
2. Then accumulate using `collections.defaultdict(list)`.
3. `assert` the two results are equal before returning.

Concept: the missing-key idiom — why `setdefault` needs one lookup and `defaultdict` shifts the default into the type.

In [17]:
from collections import defaultdict


def requests_per_path(records):
    # Version A: setdefault
    by_path_a = {}
    # TODO: fill by_path_a using dict.setdefault
    for record in records:
        by_path_a.setdefault(record.path, [])
        by_path_a[record.path].append(record.user)

    # Version B: defaultdict
    by_path_b = defaultdict(list)
    for record in records:
        by_path_b[record.path].append(record.user)

    assert by_path_a == by_path_b, "your two approaches disagree"
    return by_path_b


requests_per_path(records)

defaultdict(list,
            {'/api/orders': ['alice',
              'bob',
              'alice',
              'alice',
              'alice',
              'bob',
              'carol',
              'eve'],
             '/api/reports': ['carol', 'bob', 'carol', 'alice'],
             '/api/health': ['dave', 'dave']})

## Part 3 — Error rates (Counter + dict comprehension)

An error is any `status >= 400`.

1. Write `error_rate_by_path(records)` returning `path -> error_rate` where the rate is `errors / total_requests` for that path, as a float. Use two `collections.Counter` objects (totals and errors) and build the result with a **dict comprehension**.
2. The result must include *every* path that appears — paths with zero errors should map to `0.0`, not be missing.

Concept: `Counter` as a specialized dict, and why comprehension iteration source matters (iterate the totals, not the errors, or you'll drop clean paths).

In [18]:
from collections import Counter


def error_rate_by_path(records):
    totals = Counter([r.path for r in records])
    errors = Counter([r.path for r in records if r.status >= 400])
    # TODO: populate totals and errors from records
    
    # TODO: return {path: errors[path] / totals[path] for ... } over ALL paths
    return {path: errors[path]/totals[path] for path in totals.keys()}


error_rate_by_path(records)

{'/api/orders': 0.5, '/api/reports': 0.5, '/api/health': 0.0}

## Part 4 — Role membership questions (set algebra)

Using the `ROLES` mapping, write these functions with **set operations only** (no loops):

1. `all_users_in_records(records)` → set of every user seen in the log.
2. `users_without_role(records)` → users in the log who are in *no* role.
3. `multi_role_users()` → users who appear in *more than one* role.
4. `roles_overlap(role_a, role_b)` → the shared members of two roles.

Concept: union / intersection / difference and how to express "in no group" and "in more than one group" as set algebra rather than counting loops.

In [29]:
def all_users_in_records(records):
    # TODO: a set comprehension or set() over records
    return {r.user for r in records}


def users_without_role(records):
    # TODO: seen-users MINUS union of all role sets
    return all_users_in_records(records) - set.union(*ROLES.values()) 


def multi_role_users():
    # TODO: users appearing in >1 role. Hint: pairwise intersections,
    # or count memberships another way — but express the final answer as a set.
    from itertools import combinations
    result = set()
    for set1, set2 in combinations(ROLES.values(), 2):
        result |= set1 & set2
    return result


def roles_overlap(role_a, role_b):
    # TODO: intersection of two named roles
    return ROLES[role_a] & ROLES[role_b]


print("all:", all_users_in_records(records))
print("no role:", users_without_role(records))
print("multi role:", multi_role_users())
print("admins & analysts:", roles_overlap("admins", "analysts"))

all: {'bob', 'alice', 'eve', 'dave', 'carol'}
no role: {'eve'}
multi role: {'bob'}
admins & analysts: {'bob'}


## Part 5 — Layered request configuration (ChainMap)

Requests resolve config from three layers, highest priority first:
1. **per-request overrides** (whatever the caller passed)
2. **per-endpoint config**
3. **global defaults**

```python
DEFAULTS = {"timeout": 30, "retries": 0, "cache": False, "region": "us"}
ENDPOINT_CONFIG = {
    "/api/reports": {"timeout": 120, "cache": True},
    "/api/orders":  {"retries": 2},
}
```

Write `resolve_config(path, overrides)` that returns a **read-only** view where a lookup finds the highest-priority layer that defines the key. Use `collections.ChainMap`. The returned object must *not* allow the caller to mutate the underlying config dicts through it — wrap the result appropriately.

Also answer in a comment: if the caller later mutates `ENDPOINT_CONFIG["/api/orders"]`, does an already-returned resolver see the change? Why?

Concept: layered lookup with ChainMap, and read-only exposure (`MappingProxyType`) — the mutation question probes the difference between *copying* and *viewing*.

In [37]:
from collections import ChainMap
from types import MappingProxyType

DEFAULTS = {"timeout": 30, "retries": 0, "cache": False, "region": "us"}
ENDPOINT_CONFIG = {
    "/api/reports": {"timeout": 120, "cache": True},
    "/api/orders":  {"retries": 2},
}


def resolve_config(path, overrides):
    # TODO: build a ChainMap layering overrides > endpoint config > defaults,
    # and return it as a read-only mapping the caller cannot mutate through.
    return MappingProxyType(ChainMap(overrides, ENDPOINT_CONFIG[path], DEFAULTS))


cfg = resolve_config("/api/reports", {"retries": 5})
print(dict(cfg))
# Expected precedence: timeout=120 (endpoint), retries=5 (override),
#                      cache=True (endpoint), region='us' (default)

# Your written answer to the mutation question:
# ...

{'timeout': 120, 'retries': 5, 'cache': True, 'region': 'us'}


## Part 6 — A hashable custom key (the hash contract)

You want to deduplicate "sessions", where a session is identified by `(user, path)` — a client hitting the same endpoint counts once. Model a `SessionKey` class (NOT a namedtuple this time — write it by hand) so instances can be used as **set members** and **dict keys**.

Requirements:
- Two `SessionKey` instances with the same `user` and `path` must be equal AND have the same hash.
- Instances must be usable in a `set`.
- Make it immutable enough that its hash stays valid (block attribute reassignment, or at least don't rely on mutable state in `__hash__`).

Then write `unique_sessions(records)` returning the set of distinct `SessionKey`s.

Concept: the `__eq__`/`__hash__` contract — equal objects must hash equal, and a hashable object's hash must not change over its lifetime. This is the invariant that makes dicts and sets work at all.

In [53]:
# class SessionKey:
#     # TODO: store user and path immutably.
#     # TODO: implement __eq__ and __hash__ honouring the contract.
#     # TODO (optional but recommended): a readable __repr__.
#     def __init__(self, user, path):
#         self.user = user
#         self.path = path

#     def __eq__(self, value):
#         return self.user == value.user and self.path == value.path

#     def __hash__(self):
#         return hash((self.user, self.path))

#     def __repr__(self):
#         return f'SessionKey({self.user}, {self.path})'

def unique_sessions(records):
     # TODO: return a set of SessionKey(user, path)
     return {SessionKey(r.user, r.path) for r in records }
from typing import NamedTuple
class SessionKey(NamedTuple):
    user: str
    path: str

sessions = unique_sessions(records)
print(len(sessions), "unique sessions")
sorted(repr(s) for s in sessions)

8 unique sessions


["SessionKey(user='alice', path='/api/orders')",
 "SessionKey(user='alice', path='/api/reports')",
 "SessionKey(user='bob', path='/api/orders')",
 "SessionKey(user='bob', path='/api/reports')",
 "SessionKey(user='carol', path='/api/orders')",
 "SessionKey(user='carol', path='/api/reports')",
 "SessionKey(user='dave', path='/api/health')",
 "SessionKey(user='eve', path='/api/orders')"]

## Part 7 — Classify events with pattern matching (mapping patterns)

Convert each `Event` to a plain `dict` and write `classify(event_dict)` using `match`/`case` on the **mapping** to return a label:

- `{"status": 200 | 201, ...}` → `"ok"`
- `{"method": "DELETE", "status": 403, ...}` → `"forbidden-delete"`
- `{"status": s, ...}` where `s >= 500` → `"server-error"`
- `{"status": s, ...}` where `400 <= s < 500` → `"client-error"`
- anything else → `"unknown"`

Use mapping patterns with capture and guards. Note `case` order matters — the more specific `forbidden-delete` must precede the generic `client-error`.

Concept: `match`/`case` mapping patterns — partial matching (extra keys ignored), capture (`s`), OR-patterns (`200 | 201`), and guards (`if s >= 500`).

In [43]:
def classify(event_dict):
    match event_dict:
        # TODO: fill in the case clauses in the right order
        case {"status": 200 | 201}:
            return "ok"
        case {"method": "DELETE", "status": 403}:
            return "forbidden-delete"
        case {"status": s} if s >= 500:
            return "server-error"
        case {"status": s} if 400 <= s < 500:
            return "client-error"
        case _:
            return "unknown"


for ev in records:
    d = ev._asdict()
    print(f"{d['method']:6} {d['path']:14} {d['status']}  -> {classify(d)}")

GET    /api/orders    200  -> ok
POST   /api/orders    201  -> ok
GET    /api/orders    200  -> ok
GET    /api/reports   500  -> server-error
DELETE /api/orders    403  -> forbidden-delete
GET    /api/reports   200  -> ok
GET    /api/reports   500  -> server-error
GET    /api/health    200  -> ok
GET    /api/orders    200  -> ok
POST   /api/orders    500  -> server-error
GET    /api/orders    404  -> client-error
GET    /api/health    200  -> ok
GET    /api/reports   200  -> ok
POST   /api/orders    401  -> client-error


## Part 8 — Live dict views (predict, then verify)

Run the checks below **after** implementing the parts above. Before running, write down in the comment what you *expect*. The exercise is to correctly predict whether a view reflects later mutations.

Concept: `.keys()` / `.items()` / `.values()` are dynamic views over the dict, not snapshots.

In [35]:
rates = error_rate_by_path(records)
paths_view = rates.keys()

# PREDICT: after I add a new path below, will `paths_view` contain it?
# Your prediction: yes

rates["/api/new"] = 0.0
print("/api/new" in paths_view)   # verify your prediction

# PREDICT: does `len(paths_view)` change too?
# Your prediction: yes
print(len(paths_view))

True
4


## Self-check (run last)

These assertions verify Parts 1–7. A failure points at a concrete gap. Run this only after you've filled everything in.

In [54]:
recs = parse_log(LOG_LINES)

# Part 1: records are structured & typed
assert len(recs) == 14
assert recs[0].user == "alice" and isinstance(recs[0].status, int)

# Part 2: grouping
rp = requests_per_path(recs)
assert rp["/api/health"] == ["dave", "dave"]

# Part 3: error rates include clean paths as 0.0
er = error_rate_by_path(recs)
assert er["/api/health"] == 0.0
assert abs(er["/api/reports"] - 2 / 4) < 1e-9      # 2 of 4 report hits are 500
assert set(er) == {r.path for r in recs}           # every path present

# Part 4: set algebra
assert all_users_in_records(recs) == {"alice", "bob", "carol", "dave", "eve"}
assert users_without_role(recs) == {"eve"}
assert multi_role_users() == {"bob"}
assert roles_overlap("admins", "analysts") == {"bob"}

# Part 5: config precedence + read-only
cfg = resolve_config("/api/reports", {"retries": 5})
assert cfg["timeout"] == 120 and cfg["retries"] == 5 and cfg["cache"] is True
assert cfg["region"] == "us"
try:
    cfg["timeout"] = 1
except TypeError:
    pass
else:
    raise AssertionError("config view should be read-only")

# Part 6: hash contract
a = SessionKey("alice", "/api/orders")
b = SessionKey("alice", "/api/orders")
c = SessionKey("bob", "/api/orders")
assert a == b and hash(a) == hash(b)
assert a != c
assert len({a, b, c}) == 2
assert len(unique_sessions(recs)) == 8

# Part 7: classification
assert classify({"method": "GET", "status": 200}) == "ok"
assert classify({"method": "DELETE", "status": 403}) == "forbidden-delete"
assert classify({"method": "GET", "status": 500}) == "server-error"
assert classify({"method": "GET", "status": 404}) == "client-error"
assert classify({"method": "GET", "status": 302}) == "unknown"

print("All self-checks passed \N{WHITE HEAVY CHECK MARK}")

All self-checks passed ✅
